# NYC Taxi Data Analysis

Este notebook conecta a la base de datos PostgreSQL que se ejecuta en Docker y ejecuta diversas consultas SQL para explorar patrones de viajes en taxi (servicios, zonas, horas, ingresos, etc.). Las visualizaciones se generan con pandas, matplotlib y plotly.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import sqlalchemy
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# configure plotting aesthetics
sns.set(style="whitegrid")
%matplotlib inline

In [ ]:
# Database Connection Setup

# Adjust these parameters if necessary
DB_USER = "root"
DB_PASS = "root"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "source"

connection_string = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# quick test
with engine.connect() as conn:
    result = conn.execute(sqlalchemy.text("SELECT version();"))
    print(result.fetchone())

## 1. Trips by Month (2024)
Consulta optimizada con pruning para obtener conteo de viajes por mes en 2024.

In [ ]:
query = """
SELECT d.month, COUNT(*) as total_trips
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
WHERE f.pickup_date >= '2024-01-01' AND f.pickup_date <= '2024-12-31'
GROUP BY d.month ORDER BY d.month;
"""
df1 = pd.read_sql(query, engine)
df1

# line chart
fig = px.line(df1, x='month', y='total_trips', title='Viajes por mes 2024')
fig.show()

(2024): Esta consulta contabiliza el volumen total de servicios realizados en cada mes del año 2024 para identificar los periodos de mayor demanda.

## 2. Trips by Service Type and Month
Agrupa por tipo de servicio y mes.

In [ ]:
query = """
SELECT s.service_type, d.month, COUNT(*) as total_trips
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_service_type s ON f.service_type_key = s.service_type_key
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
GROUP BY s.service_type, d.month ORDER BY d.month;
"""
df2 = pd.read_sql(query, engine)
df2

fig = px.line(df2, x='month', y='total_trips', color='service_type',
              title='Viajes por tipo de servicio y mes')
fig.show()

Este análisis desglosa la cantidad de viajes mensuales diferenciando entre los tipos de servicio disponibles para entender la cuota de mercado. Los yellow suelen tener mas viajes

## 3. Top 10 Pickup Zones (2024)

In [ ]:
query = """
SELECT z.zone_name, COUNT(*) as total_trips
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pickup_location_key = z.zone_key
WHERE f.pickup_date >= '2024-01-01' AND f.pickup_date < '2025-01-01'
GROUP BY 1 ORDER BY total_trips DESC LIMIT 10;
"""
df3 = pd.read_sql(query, engine)
df3

fig = px.bar(df3, x='total_trips', y='zone_name', orientation='h',
             title='Top 10 zonas de pickup 2024')
fig.show()

(2024) El script identifica los diez sectores geográficos donde se originó la mayor cantidad de viajes durante el transcurso del año 2024.

## 4. Top 10 Dropoff Zones (2024)

In [ ]:
query = """
SELECT z.zone_name, COUNT(*) as total_trips
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.dropoff_location_key = z.zone_key
WHERE f.pickup_date >= '2024-01-01' AND f.pickup_date < '2025-01-01'
GROUP BY 1 ORDER BY total_trips DESC LIMIT 10;
"""
df4 = pd.read_sql(query, engine)
df4

fig = px.bar(df4, x='total_trips', y='zone_name', orientation='h',
             title='Top 10 zonas de dropoff 2024')
fig.show()

(2024) Esta métrica revela los diez destinos más frecuentes elegidos por los usuarios finales a lo largo del periodo anual analizado. Permite contrastar si los centros de mayor origen de viajes coinciden geográficamente con los puntos de llegada con mayor densidad de tráfico

## 5. Top 5 Boroughs by Month (pickup)
Ranking mensual de boroughs.

In [ ]:
query = """
SELECT * FROM (
    SELECT z.borough, EXTRACT(MONTH FROM f.pickup_date) as month, COUNT(*) as total_trips,
    RANK() OVER(PARTITION BY EXTRACT(MONTH FROM f.pickup_date) ORDER BY COUNT(*) DESC) as rnk
    FROM analytics_gold.fct_trips f
    JOIN analytics_gold.dim_zone z ON f.pickup_location_key = z.zone_key
    WHERE f.pickup_date >= '2024-01-01' AND f.pickup_date < '2025-01-01'
    GROUP BY 1, 2
) t WHERE rnk <= 5;
"""
df5 = pd.read_sql(query, engine)
df5

# pivot for display
pivot5 = df5.pivot(index='month', columns='borough', values='total_trips')
pivot5

Utiliza funciones de ventana para clasificar los cinco distritos con mayor actividad de salida en cada mes de manera independiente. Facilita la observación de cambios en la jerarquía de demanda territorial y cómo la importancia de cada distrito fluctúa temporalmente.

## 6. Peak Hours by Day of Week
Top 5 horas pico por día de la semana.

In [ ]:
query = """
SELECT * FROM (
    SELECT d.day_of_week, t.hour, COUNT(*) as total_trips,
    RANK() OVER(PARTITION BY d.day_of_week ORDER BY COUNT(*) DESC) as rnk
    FROM analytics_gold.fct_trips f
    JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
    JOIN analytics_gold.dim_time t ON f.pickup_time_key = t.time_key
    GROUP BY d.day_of_week, t.hour
) hr WHERE rnk <= 5;
"""
df6 = pd.read_sql(query, engine)
df6

# heatmap preparation
heat6 = df6.pivot(index='day_of_week', columns='hour', values='total_trips')
plt.figure(figsize=(10,6))
sns.heatmap(heat6, annot=True, fmt="g", cmap="YlOrRd")
plt.title('Peak hours by day of week')
plt.show()

El análisis extrae las cinco franjas horarias de mayor congestión y demanda para cada día individual de la semana laboral y fines de semana. De todas las semanas

## 7. Trip Distribution by Day of Week

In [ ]:
query = """
SELECT d.day_of_week, COUNT(*) as total_trips
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
GROUP BY d.day_of_week ORDER BY total_trips DESC;
"""
df7 = pd.read_sql(query, engine)
df7

fig = px.bar(df7, x='day_of_week', y='total_trips', title='Distribución de viajes por día de semana')
fig.show()

Esta consulta genera un ranking de los días de la semana basándose en el volumen acumulado de viajes registrados en el modelo

## 8. Total Revenue by Month

In [ ]:
query = """
SELECT d.month, SUM(f.total_amount) as total_revenue
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
GROUP BY d.month ORDER BY d.month;
"""
df8 = pd.read_sql(query, engine)
df8

fig = px.line(df8, x='month', y='total_revenue', title='Ingreso total por mes')
fig.show()

Se calcula la recaudación bruta mensual sumando el monto total facturado por todos los servicios de transporte realizados con éxito.

## 9. Revenue by Service Type and Month (2024)

In [ ]:
query = """
SELECT f.service_type_key, EXTRACT(MONTH FROM f.pickup_date) as month, SUM(f.total_amount) as total_revenue
FROM analytics_gold.fct_trips f
WHERE f.pickup_date >= '2024-01-01' AND f.pickup_date < '2025-01-01'
GROUP BY 1, 2 ORDER BY 2;
"""
df9 = pd.read_sql(query, engine)
df9

fig = px.bar(df9, x='month', y='total_revenue', color='service_type_key',
             title='Ingreso por tipo de servicio y mes (2024)', barmode='group')
fig.show()

Este informe compara los ingresos generados por los servicios amarillo y verde de manera mensual durante el transcurso del año 2024. Permite observar cuál categoría de taxi contribuye con mayor peso al flujo financiero total de la operación de transporte.

## 10. Average Tip Percentage by Month (2024)

In [ ]:
query = """
SELECT EXTRACT(MONTH FROM f.pickup_date) as month, AVG(f.tip_amount / NULLIF(f.fare_amount,0)) * 100 as avg_tip_pct
FROM analytics_gold.fct_trips f
WHERE f.pickup_date >= '2024-01-01' AND f.pickup_date < '2025-01-01'
GROUP BY 1 ORDER BY 1;
"""
df10 = pd.read_sql(query, engine)
df10

fig = px.line(df10, x='month', y='avg_tip_pct', title='Tip % promedio por mes (2024)')
fig.show()

(2024): Calcula el porcentaje promedio de propina dejado por los usuarios en relación con la tarifa base facturada durante cada mes. Sirve como un indicador indirecto de la satisfacción del cliente

## 11. Average Tip Percentage by Borough and Month (2024)

In [ ]:
query = """
SELECT z.borough, EXTRACT(MONTH FROM f.pickup_date) as month, AVG(f.tip_amount / NULLIF(f.fare_amount,0)) * 100 as avg_tip_pct
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pickup_location_key = z.zone_key
WHERE f.pickup_date >= '2024-01-01' AND f.pickup_date < '2025-01-01'
GROUP BY 1, 2 ORDER BY 2;
"""
df11 = pd.read_sql(query, engine)
df11

fig = px.line(df11, x='month', y='avg_tip_pct', color='borough',
              title='Tip % por borough y mes (2024)')
fig.show()

(2024): Evalúa la generosidad de los pasajeros desglosada por distrito geográfico y mes durante el año calendario 2024 para detectar tendencias.

## 12. Top 10 Zones by Total Revenue (pickup)

In [ ]:
query = """
SELECT z.zone_name, SUM(f.total_amount) as total_revenue
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pickup_location_key = z.zone_key
WHERE f.pickup_date >= '2022-01-01'
GROUP BY 1 ORDER BY total_revenue DESC LIMIT 10;
"""
df12 = pd.read_sql(query, engine)
df12

fig = px.bar(df12, x='total_revenue', y='zone_name', orientation='h',
             title='Top 10 zonas por ingreso total (pickup)')
fig.show()

Determina las diez zonas geográficas que reportan los mayores ingresos totales acumulados desde el punto de inicio del viaje realizado. Ayuda a identificar los sectores de la ciudad que generan el mayor valor económico y volumen de facturación para los conductores.

## 13. Top 10 Zones by Tip Percentage (pickup) with N>=100

In [ ]:
query = """
SELECT z.zone_name, AVG(f.tip_amount / NULLIF(f.fare_amount,0)) * 100 as avg_tip_pct
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pickup_location_key = z.zone_key
WHERE f.pickup_date >= '2022-01-01'
GROUP BY 1 HAVING COUNT(*) >= 100
ORDER BY avg_tip_pct DESC LIMIT 10;
"""
df13 = pd.read_sql(query, engine)
df13

fig = px.bar(df13, x='avg_tip_pct', y='zone_name', orientation='h',
             title='Top 10 zonas por tip % (>=100 viajes)')
fig.show()

(N=100): Clasifica los sectores con mejores porcentajes de propina, filtrando solo aquellos con al menos cien viajes registrados para asegurar robustez

## 14. Cash vs Card Payment Comparison

In [ ]:
query = """
SELECT f.payment_type_key, COUNT(*) as total_trips, SUM(f.total_amount) as total_revenue,
AVG(f.tip_amount / NULLIF(f.fare_amount,0)) * 100 as avg_tip_pct
FROM analytics_gold.fct_trips f
WHERE f.pickup_date >= '2022-01-01'
GROUP BY 1;
"""
df14 = pd.read_sql(query, engine)
df14

# bar charts for trips and revenue
fig = px.bar(df14, x='payment_type_key', y=['total_trips','total_revenue'],
             title='Comparación efectivo vs tarjeta', barmode='group')
fig.show()

fig2 = px.bar(df14, x='payment_type_key', y='avg_tip_pct',
              title='Porcentaje de propina por método de pago')
fig2.show()

Realiza una comparativa integral entre los métodos de pago en efectivo y tarjeta considerando volumen de viajes, ingresos y propinas. Revela la tendencia de los usuarios hacia la digitalización de los pagos y cómo el método elegido influye directamente en el porcentaje de propina.

## 15. Average Trip Duration by Month (2024)

In [ ]:
query = """
SELECT EXTRACT(MONTH FROM f.pickup_date) as month, AVG(f.trip_duration_minutes) as avg_duration
FROM analytics_gold.fct_trips f
WHERE f.pickup_date >= '2024-01-01' AND f.pickup_date < '2025-01-01'
GROUP BY 1 ORDER BY 1;
"""
df15 = pd.read_sql(query, engine)
df15

fig = px.line(df15, x='month', y='avg_duration', title='Duración promedio por mes (2024)')
fig.show()

Calcula el tiempo medio de duración de los traslados expresado en minutos para cada uno de los meses dentro del modelo gold. Ayuda a inferir niveles de congestión vial

## 16. Average Trip Distance by Month (2024)

In [ ]:
query = """
SELECT EXTRACT(MONTH FROM f.pickup_date) as month, AVG(f.trip_distance) as avg_distance
FROM analytics_gold.fct_trips f
WHERE f.pickup_date >= '2024-01-01' AND f.pickup_date < '2025-01-01'
GROUP BY 1 ORDER BY 1;
"""
df16 = pd.read_sql(query, engine)
df16

fig = px.line(df16, x='month', y='avg_distance', title='Distancia promedio por mes (2024)')
fig.show()

(2024): Esta consulta analiza la longitud promedio de los viajes realizados mensualmente durante el año 2024 para identificar patrones de distancia. Permite determinar si los trayectos tienden a ser más largos o cortos dependiendo de las condiciones climáticas o eventos de la época.

## 17. Average Speed by Borough and Hour

In [ ]:
query = """
SELECT z.borough, f.pickup_time_key / 60 as hour_slot, 
AVG(f.trip_distance / NULLIF(f.trip_duration_minutes / 60, 0)) as avg_speed_mph
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pickup_location_key = z.zone_key
WHERE f.pickup_date >= '2022-01-01'
GROUP BY 1, 2;
"""
df17 = pd.read_sql(query, engine)
df17

heat17 = df17.pivot(index='borough', columns='hour_slot', values='avg_speed_mph')
plt.figure(figsize=(12,6))
sns.heatmap(heat17, cmap='coolwarm')
plt.title('Velocidad promedio por borough y franja horaria')
plt.show()

Estima la velocidad media de circulación en millas por hora cruzando información de distritos geográficos y franjas horarias específicas. Es un indicador directo del estado del tráfico y la fluidez vial en diferentes puntos estratégicos de la ciudad de Nueva York.

## 18. Duration Percentiles by Borough

In [ ]:
query = """
SELECT z.borough, 
percentile_cont(0.5) WITHIN GROUP (ORDER BY f.trip_duration_minutes) as p50,
percentile_cont(0.9) WITHIN GROUP (ORDER BY f.trip_duration_minutes) as p90
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pickup_location_key = z.zone_key
WHERE f.pickup_date >= '2022-01-01'
GROUP BY 1;
"""
df18 = pd.read_sql(query, engine)
df18

fig = px.box(df18, y=['p50','p90'], points='all',
             title='Percentiles de duración por borough')
fig.show()

Obtiene la mediana y el percentil noventa de la duración de los viajes para caracterizar el comportamiento estadístico de cada distrito. El percentil 90 es especialmente útil para entender los tiempos de viaje en condiciones desfavorables o de tráfico inusualmente intenso.

## 19. Top 10 Zones by P90 Duration

In [ ]:
query = """
SELECT z.zone_name, percentile_cont(0.9) WITHIN GROUP (ORDER BY f.trip_duration_minutes) as p90
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pickup_location_key = z.zone_key
WHERE f.pickup_date >= '2022-01-01'
GROUP BY 1 ORDER BY p90 DESC LIMIT 10;
"""
df19 = pd.read_sql(query, engine)
df19

fig = px.bar(df19, x='p90', y='zone_name', orientation='h',
             title='Top 10 zonas por p90 de duración')
fig.show()

Identifica los diez sectores donde los viajes tienden a sufrir los mayores retrasos o duraciones extremas según el percentil noventa. Ayuda a señalar puntos críticos de congestión severa

## 20. Top 10 Borough-to-Borough Routes

In [ ]:
query = """
SELECT zp.borough as p_bor, zd.borough as d_bor, COUNT(*) as total
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone zp ON f.pickup_location_key = zp.zone_key
JOIN analytics_gold.dim_zone zd ON f.dropoff_location_key = zd.zone_key
WHERE f.pickup_date >= '2022-01-01'
GROUP BY 1, 2 ORDER BY total DESC LIMIT 10;
"""
df20 = pd.read_sql(query, engine)
df20

# sankey diagram
fig = px.sankey(df20, node = dict(label=list(pd.unique(df20[['p_bor','d_bor']].values.ravel()))),
                link = dict(source=[list(pd.unique(df20[['p_bor','d_bor']].values.ravel())).index(x) for x in df20['p_bor']],
                            target=[list(pd.unique(df20[['p_bor','d_bor']].values.ravel())).index(x) for x in df20['d_bor']],
                            value=df20['total']))
fig.update_layout(title='Top 10 rutas borough->borough')
fig.show()

Analiza los flujos de movimiento principales entre los diferentes distritos de la ciudad basándose en el volumen total de viajes completados. Revela las rutas de mayor demanda